In [36]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

_nb_dir = Path().resolve()
_root = next(
    (p for p in [_nb_dir] + list(_nb_dir.parents) if (p / "pyproject.toml").exists()),
    _nb_dir.parent,
)
_src = str(_root / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

load_dotenv(_root / ".env", override=True)

print(f"프로젝트 루트: {_root}")
print(f"Python 경로 등록: {_src}")

프로젝트 루트: C:\Users\user\dev\catcher-llm
Python 경로 등록: C:\Users\user\dev\catcher-llm\src


In [37]:
import time
from catcher_llm.config.settings import Settings
from catcher_llm.services.consumption_feedback.daily_feedback import (
    generate_daily_feedback,
    make_daily_feedback_input,
)

TEST_CASES = [
    {"member_id": 1, "analysis_date": "2026-04-29", "previous_date": "2026-04-28"},
]

base_settings = Settings()
shared_inputs = []

print("🔄 공통 파이프라인 입력 생성 중 (분석 → 해석 → RAG)...")
print("   이 단계는 1회만 실행합니다. 약 30~90초 소요됩니다.\n")

for case in TEST_CASES:
    date_label = case["analysis_date"]
    print(f"  📅 {date_label} 처리 중...")
    try:
        baseline = generate_daily_feedback(
            member_id=case["member_id"],
            analysis_date=case["analysis_date"],
            previous_date=case["previous_date"],
            settings=base_settings,
        )
        if baseline.error:
            raise Exception(f"파이프라인 실행 오류: {baseline.error}")
        feedback_input = make_daily_feedback_input(
            user_data=baseline.daily_analysis,
            interpretation_result=baseline.interpretation_result,
            advice_contexts=baseline.retrieved_contexts,
            user_profile=baseline.user_profile,
            memory_context=baseline.memory_context,
        )
        shared_inputs.append({
            "date": date_label,
            "feedback_input": feedback_input,
            "baseline_feedback": baseline.feedback,
        })
        print(f"  ✅ 완료")
    except Exception as e:
        print(f"  ❌ 오류: {e}")

print(f"\n공통 입력 생성 완료: {len(shared_inputs)}개 날짜")

🔄 공통 파이프라인 입력 생성 중 (분석 → 해석 → RAG)...
   이 단계는 1회만 실행합니다. 약 30~90초 소요됩니다.

  📅 2026-04-29 처리 중...
  ✅ 완료

공통 입력 생성 완료: 1개 날짜


In [38]:
import re
from pydantic import SecretStr
from langchain_openai import ChatOpenAI
from catcher_llm.chains.consumption_feedback import build_daily_feedback_chain
from catcher_llm.schemas.consumption_feedback.daily import DailyFeedbackResult

MODELS_TO_COMPARE_PROJ = [
    "gpt-4o-mini",
    "gpt-4.1-nano",
    "gpt-4.1-mini",
    "gpt-5-nano",
    "gpt-5-mini",
]

def _get_temperature(model):
    return 1.0 if "gpt-5" in model else 0.0

def run_feedback_for_model(model, feedback_input, analysis_date):
    """지정 모델로 최종 피드백 체인만 실행하고 결과를 반환한다."""
    llm = ChatOpenAI(
        api_key=SecretStr(os.getenv("OPENAI_API_KEY", "")),
        model=model,
        temperature=_get_temperature(model),
    )
    chain = build_daily_feedback_chain(llm=llm)
    t0 = time.perf_counter()
    result = chain.invoke(feedback_input)
    latency = round(time.perf_counter() - t0, 3)
    if isinstance(result, DailyFeedbackResult):
        return {
            "model": model,
            "analysis_date": analysis_date,
            "latency_sec": latency,
            "scolding_message": result.scolding_message,
            "tomorrow_mission": result.tomorrow_mission,
            "summary_title": result.summary_title,
        }
    raise ValueError(f"Unexpected result type: {type(result)}")

print("피드백 체인 실행 함수 정의 완료 ✅")

피드백 체인 실행 함수 정의 완료 ✅


In [39]:
all_results = []
run_errors = []

print("🔬 모델별 피드백 체인 실행 중...")

for entry in shared_inputs:
    date_label = entry["date"]
    feedback_input = entry["feedback_input"]
    print(f"\n  📅 {date_label}")
    for model in MODELS_TO_COMPARE_PROJ:
        print(f"    {model:<20}", end=" ", flush=True)
        try:
            result = run_feedback_for_model(model, feedback_input, date_label)
            all_results.append(result)
            print(f"✅ {result['latency_sec']:.2f}초 | 피드백 {len(result['scolding_message'])}자")
        except Exception as e:
            run_errors.append({"model": model, "date": date_label, "error": str(e)[:120]})
            print(f"❌ {str(e)[:80]}")

print(f"\n실행 완료: {len(all_results)}개 성공 / {len(run_errors)}개 실패")

🔬 모델별 피드백 체인 실행 중...

  📅 2026-04-29
    gpt-4o-mini          ✅ 12.02초 | 피드백 233자
    gpt-4.1-nano         ✅ 3.48초 | 피드백 310자
    gpt-4.1-mini         ✅ 7.51초 | 피드백 289자
    gpt-5-nano           ✅ 62.90초 | 피드백 264자
    gpt-5-mini           ✅ 37.88초 | 피드백 187자

실행 완료: 5개 성공 / 0개 실패


In [40]:
import json
import re

def run_llm_comparative_eval(results):
    evaluator_llm = ChatOpenAI(model="gpt-4o", temperature=0.2) # 통찰력을 위해 4o 권장
    
    # 1. 모든 모델의 결과물을 하나의 텍스트로 통합
    candidates_text = ""
    for i, res in enumerate(results):
        candidates_text += f"""
---
[후보 {i+1}]
모델명: {res['model']}
제목: {res['summary_title']}
피드백: {res['scolding_message']}
미션: {res['tomorrow_mission']}
"""

    # 2. LLM에게 자율적인 기준 설정과 순위 산정 요청
    system_prompt = """
당신은 최고의 서비스 기획자이자 언어 모델 평가 전문가입니다.
제시된 여러 모델의 답변들을 비교하여, 어떤 모델이 '가장 사용자 경험(UX)이 뛰어난 답변'을 작성했는지 평가해야 합니다.

당신은 스스로 '좋은 피드백'의 기준(예: 분석의 깊이, 가독성, 동기부여 등)을 정의하고, 
그 기준에 따라 모든 후보의 순위를 매기세요.

결과는 반드시 아래 JSON 형식을 지켜주세요:
{
    "evaluation_criteria": ["본인이 세운 기준 1", "기준 2", ...],
    "ranking": [
        {"rank": 1, "model": "모델명", "reason": "선정 이유"},
        {"rank": 2, "model": "모델명", "reason": "선정 이유"}
    ],
    "best_overall": "최고의 모델명"
}
"""

    user_prompt = f"""
다음은 동일한 입력 데이터에 대해 서로 다른 AI 모델이 생성한 결과물들입니다.
이들을 정밀하게 비교하여 순위를 매겨주세요.

{candidates_text}
"""

    print("⚖️  LLM이 스스로 기준을 세워 모델 성능을 비교 분석 중입니다...")
    
    try:
        response = evaluator_llm.invoke([
            ("system", system_prompt),
            ("user", user_prompt)
        ])
        
        # JSON 응답 파싱
        eval_result = json.loads(re.search(r"\{.*\}", response.content, re.DOTALL).group())
        
        # 결과 출력
        print("\n🔍 [LLM이 설정한 평가 기준]")
        for criterion in eval_result['evaluation_criteria']:
            print(f" - {criterion}")
            
        print("\n🏆 [모델별 최종 랭킹]")
        for item in eval_result['ranking']:
            print(f"{item['rank']}위: {item['model']} | {item['reason']}")
            
        return eval_result

    except Exception as e:
        print(f"❌ 평가 도중 오류 발생: {e}")
        return None

# 평가 실행
comparison_report = run_llm_comparative_eval(all_results)

⚖️  LLM이 스스로 기준을 세워 모델 성능을 비교 분석 중입니다...

🔍 [LLM이 설정한 평가 기준]
 - 분석의 깊이
 - 가독성
 - 동기부여
 - 구체적인 조언
 - 사용자 맞춤화

🏆 [모델별 최종 랭킹]
1위: gpt-5-nano | 분석의 깊이가 뛰어나고, 사용자에게 맞춤화된 조언을 제공하며, 동기부여 요소가 강합니다. 구체적인 행동 계획을 제시하여 실천 가능성을 높였습니다.
2위: gpt-4.1-nano | 절약의 중요성을 강조하며, 소비 패턴의 변화에 대한 경고와 함께 균형 잡힌 소비 습관을 제안합니다. 동기부여가 잘 되어 있으며, 구체적인 조언을 제공합니다.
3위: gpt-5-mini | 지출 억제 노력에 대한 칭찬과 함께 구체적인 행동 계획을 제시합니다. 다만, 사용자 맞춤화 측면에서 다소 일반적인 조언에 그칩니다.
4위: gpt-4o-mini | 소비 패턴의 변화에 대한 경고와 함께 구체적인 계획을 제시하지만, 분석의 깊이와 사용자 맞춤화 측면에서 다소 부족합니다.
5위: gpt-4.1-mini | 소비 패턴의 변화에 대한 분석은 있지만, 다른 후보들에 비해 동기부여 요소가 약하고, 조언이 다소 일반적입니다.


- LLM 자율 평가를 믿어야 하는 이유: 사용자는 %가 들어갔는지보다 "내 소비를 진짜 걱정해 주는지"에서 가치를 느낍니다. 따라서 모델 성능의 상한선(Ceiling)은 LLM 평가 결과로 결정해야 합니다.

- 규칙 기반 점수를 써야 하는 이유: 이건 모델의 하한선(Floor)입니다. 시스템 UI가 350자 이상을 요구하는데 모델이 짧게 대답하면 화면이 깨질 수 있습니다. 즉, 규칙 점수는 '품질'이 아니라 '동작 여부'를 확인하는 용도입니다.

# 프롬프트 고쳐보기 

In [41]:
def run_llm_comparative_eval_v2(results, heuristics_table):
    evaluator_llm = ChatOpenAI(model="gpt-4o", temperature=0.1)
    
    # 모델별 규칙 점수를 텍스트로 정리
    candidates_text = ""
    for i, res in enumerate(results):
        # 해당 모델의 규칙 점수 찾기 (섹션 2 표 데이터 활용)
        h_score = next((item for item in heuristics_table if item['모델'] == res['model']), {})
        
        candidates_text += f"""
---
[후보 {i+1}]
- 모델명: {res['model']}
- 규칙 점수 (시스템 준수율): 정확성 {h_score.get('정확성(0-1)', 'N/A')}, 형식 {h_score.get('형식(0-1)', 'N/A')}
- 답변 제목: {res['summary_title']}
- 답변 피드백: {res['scolding_message']}
- 답변 미션: {res['tomorrow_mission']}
"""

    system_prompt = """
당신은 최고의 데이터 분석가이자 소비자 심리 전문가입니다. 
단순한 텍스트 비교를 넘어, 아래 4가지 고도화된 기준에 따라 모델들의 성능을 판별하세요.

1. 데이터 인사이트(Data Insight): 수치 간의 상관관계를 분석하여 소비 원인을 정확히 짚어내는가?
2. 전략적 넛지(Strategic Nudge): 사용자가 거부감 없이 즉각 실행할 수 있는 구체적이고 작은 행동 단위를 제시하는가?
3. 심리적 설득력(Persuasiveness): 사용자의 소비 페르소나를 반영한 맞춤형 톤앤매너를 구사하는가?
4. 구조적 명료성(Structural Clarity): 정보의 우선순위가 명확하여 분석 내용이 뇌에 즉각적으로 각인되는가?
"""

    user_prompt = f"""
다음은 동일한 입력 데이터에 대해 서로 다른 AI 모델이 생성한 결과물들입니다.
이들을 정밀하게 비교하여 순위를 매겨주세요.

{candidates_text}
"""

    print("⚖️  LLM이 스스로 기준을 세워 모델 성능을 비교 분석 중입니다...")
    
    try:
        response = evaluator_llm.invoke([
            ("system", system_prompt),
            ("user", user_prompt)
        ])
        
        # JSON 응답 파싱
        eval_result = json.loads(re.search(r"\{.*\}", response.content, re.DOTALL).group())
        
        # 결과 출력
        print("\n🔍 [LLM이 설정한 평가 기준]")
        for criterion in eval_result['evaluation_criteria']:
            print(f" - {criterion}")
            
        print("\n🏆 [모델별 최종 랭킹]")
        for item in eval_result['ranking']:
            print(f"{item['rank']}위: {item['model']} | {item['reason']}")
            
        return eval_result

    except Exception as e:
        print(f"❌ 평가 도중 오류 발생: {e}")
        return None

# 평가 실행
comparison_report = run_llm_comparative_eval(all_results)

⚖️  LLM이 스스로 기준을 세워 모델 성능을 비교 분석 중입니다...

🔍 [LLM이 설정한 평가 기준]
 - 분석의 깊이
 - 가독성
 - 동기부여
 - 구체적인 조언

🏆 [모델별 최종 랭킹]
1위: gpt-5-nano | 분석의 깊이가 깊고, 사용자의 이름을 언급하여 개인화된 느낌을 줍니다. 또한, 긍정적인 동기부여와 함께 구체적인 조언을 제공하여 사용자 경험이 뛰어납니다.
2위: gpt-4.1-nano | 절약의 중요성을 강조하며, 소비 패턴의 변화에 대한 경고와 함께 구체적인 조언을 제공합니다. 동기부여 측면에서도 좋은 평가를 받을 수 있습니다.
3위: gpt-5-mini | 쇼핑 비중의 문제점을 잘 짚어내고, 구체적인 행동 계획을 제시합니다. 다만, 동기부여 측면에서 다소 부족합니다.
4위: gpt-4o-mini | 소비 패턴의 변화에 대한 분석은 좋으나, 동기부여가 부족하고 조언이 다소 일반적입니다.
5위: gpt-4.1-mini | 분석은 적절하나, 다른 모델들에 비해 동기부여와 구체적인 조언이 부족합니다.


상대 평가 vs 절대 평가:

자율 평가: 여러 모델의 답변을 한자리에 모아놓고 "얘보다 얘가 낫네"라고 순위를 매깁니다.

이 코드(LangSmith): 각 답변을 하나씩 따로따로 정해진 시험지에 따라 채점합니다.

기준의 고정성:

자율 평가: LLM이 답변들을 보고 "이 상황엔 '가독성'이 중요하겠어"라며 기준을 즉석에서 세웁니다.

이 코드: 코딩된 3가지 기준(정확성, 구용성, 형식) 안에서만 점수가 나옵니다.

# 클로드

In [42]:
import os
import time
import re
from pydantic import SecretStr
# Anthropic 모델을 위한 클래스 임포트
from langchain_anthropic import ChatAnthropic 
from catcher_llm.chains.consumption_feedback import build_daily_feedback_chain
from catcher_llm.schemas.consumption_feedback.daily import DailyFeedbackResult

# 대상을 Claude 모델 2종으로 변경
MODELS_TO_COMPARE_PROJ = [
    "claude-3-5-sonnet-20241022", # Sonnet 3.5 New
    "claude-3-5-haiku-20241022"   # Haiku 3.5
]

def _get_temperature(model):
    # Claude 모델에 맞게 온도 설정 (필요 시 조정)
    return 0.0

def run_feedback_for_model(model, feedback_input, analysis_date):
    """지정 모델(Claude)로 최종 피드백 체인만 실행하고 결과를 반환한다."""
    
    # Anthropic API를 사용하도록 LLM 인스턴스 생성 방식 변경
    llm = ChatAnthropic(
        model=model,
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY", ""),
        temperature=_get_temperature(model),
    )
    
    chain = build_daily_feedback_chain(llm=llm)
    
    t0 = time.perf_counter()
    result = chain.invoke(feedback_input)
    latency = round(time.perf_counter() - t0, 3)
    
    if isinstance(result, DailyFeedbackResult):
        return {
            "model": model,
            "analysis_date": analysis_date,
            "latency_sec": latency,
            "scolding_message": result.scolding_message,
            "tomorrow_mission": result.tomorrow_mission,
            "summary_title": result.summary_title,
        }
    raise ValueError(f"Unexpected result type: {type(result)}")

print("Claude 모델용 피드백 체인 실행 함수 정의 완료 ✅")

Claude 모델용 피드백 체인 실행 함수 정의 완료 ✅


In [43]:
uv pip install langchain-anthropic

Note: you may need to restart the kernel to use updated packages.


c:\Users\user\dev\catcher-llm\.venv\Scripts\python.exe: No module named uv


In [44]:
llm = ChatAnthropic(
    model=model,
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY", ""),
    # 베이스 URL을 공식 주소로 명시적으로 고정
    anthropic_api_url="https://api.anthropic.com", 
    temperature=_get_temperature(model),
)

In [45]:
all_results = []
run_errors = []

print("🔬 Claude 모델별 피드백 체인 실행 중 (Sonnet vs Haiku)...")

for entry in shared_inputs:
    date_label = entry["date"]
    feedback_input = entry["feedback_input"]
    print(f"\n   📅 분석 기준일: {date_label}")
    
    for model in MODELS_TO_COMPARE_PROJ:
        # 모델 출력 포맷팅을 위해 이름을 왼쪽 정렬 (25자)
        print(f"    {model:<25}", end=" ", flush=True)
        try:
            # 앞서 정의한 ChatAnthropic 기반의 run_feedback_for_model 호출
            result = run_feedback_for_model(model, feedback_input, date_label)
            all_results.append(result)
            
            # 지연 시간과 결과물 길이 출력
            print(f"✅ {result['latency_sec']:.2f}초 | 피드백 {len(result['scolding_message']):>3}자")
            
        except Exception as e:
            # 오류 발생 시 기록 (API 키 미설정, rate limit 등)
            run_errors.append({"model": model, "date": date_label, "error": str(e)[:120]})
            print(f"❌ 오류 발생: {str(e)[:80]}")

print("-" * 60)
print(f"📊 실행 완료: {len(all_results)}개 성공 / {len(run_errors)}개 실패")

🔬 Claude 모델별 피드백 체인 실행 중 (Sonnet vs Haiku)...

   📅 분석 기준일: 2026-04-29
    claude-3-5-sonnet-20241022 ❌ 오류 발생: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'messag
    claude-3-5-haiku-20241022 ❌ 오류 발생: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_error', 'messag
------------------------------------------------------------
📊 실행 완료: 0개 성공 / 2개 실패


In [46]:
import json
import re

def run_claude_comparative_eval(results, heuristics_table):
    # 1. 심판 모델 설정
    evaluator_llm = ChatOpenAI(model="gpt-4o", temperature=0.1)
    
    # 2. Claude 모델(Sonnet & Haiku) 결과만 필터링
    # 앞선 코드에서 사용된 정확한 모델명을 리스트에 담습니다.
    target_models = ["claude-3-5-sonnet-20240620", "claude-3-haiku-20240307"]
    claude_results = [res for res in results if res['model'] in target_models]
    
    if len(claude_results) < 2:
        print(f"❌ 비교할 Claude 데이터가 부족합니다. (현재: {[r['model'] for r in claude_results]})")
        print("참고: 404 오류 등으로 all_results에 Claude 데이터가 쌓이지 않았을 수 있습니다.")
        return None

    # 3. 비교 대상 텍스트 구성 (기존 로직 유지)
    candidates_text = ""
    for i, res in enumerate(claude_results):
        # 해당 모델의 규칙 점수 매칭 (섹션 2 데이터 활용)
        h_score = next((item for item in heuristics_table if item['모델'] == res['model']), {})
        
        candidates_text += f"""
---
[후보 {i+1}]
- 모델명: {res['model']}
- 규칙 점수: 정확성 {h_score.get('정확성(0-1)', 'N/A')}, 형식 {h_score.get('형식(0-1)', 'N/A')}
- 답변 제목: {res['summary_title']}
- 답변 피드백: {res['scolding_message']}
- 답변 미션: {res['tomorrow_mission']}
"""

    # 4. 요청하신 4가지 고도화된 기준이 담긴 프롬프트 (수정 없음)
    system_prompt = """
당신은 최고의 데이터 분석가이자 소비자 심리 전문가입니다. 
단순한 텍스트 비교를 넘어, 아래 4가지 고도화된 기준에 따라 모델들의 성능을 판별하세요.

1. 데이터 인사이트(Data Insight): 수치 간의 상관관계를 분석하여 소비 원인을 정확히 짚어내는가?
2. 전략적 넛지(Strategic Nudge): 사용자가 거부감 없이 즉각 실행할 수 있는 구체적이고 작은 행동 단위를 제시하는가?
3. 심리적 설득력(Persuasiveness): 사용자의 소비 페르소나를 반영한 맞춤형 톤앤매너를 구사하는가?
4. 구조적 명료성(Structural Clarity): 정보의 우선순위가 명확하여 분석 내용이 뇌에 즉각적으로 각인되는가?
"""

    user_prompt = f"""
다음은 동일한 입력 데이터에 대해 두 Claude 모델이 생성한 결과물입니다.
이들을 정밀하게 비교하여 순위를 매겨주세요.

{candidates_text}
"""

    print(f"⚖️  Claude 1:1 비교 분석 중... ({target_models[0]} vs {target_models[1]})")
    
    try:
        response = evaluator_llm.invoke([
            ("system", system_prompt),
            ("user", user_prompt)
        ])
        
        # JSON 응답 파싱
        eval_result = json.loads(re.search(r"\{.*\}", response.content, re.DOTALL).group())
        
        # 결과 출력 (형식 유지)
        print("\n🔍 [전문가 평가 기준]")
        for criterion in eval_result.get('evaluation_criteria', ["데이터 인사이트", "전략적 넛지", "심리적 설득력", "구조적 명료성"]):
            print(f" - {criterion}")
            
        print("\n🏆 [Claude 최종 랭킹]")
        for item in eval_result['ranking']:
            print(f"{item['rank']}위: {item['model']} | {item['reason']}")
            
        return eval_result

    except Exception as e:
        print(f"❌ 평가 도중 오류 발생: {e}")
        return None

# 평가 실행
# 이 함수를 호출할 때 results와 섹션2의 테이블 데이터를 넣어주세요.
comparison_report = run_claude_comparative_eval(all_results, section2_table_data)

NameError: name 'section2_table_data' is not defined

# gpt랑 클로드랑 비교하기 

In [ ]:
import json
import re
import time
import os
from pydantic import SecretStr
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

# 1. 비교 대상 모델 리스트 설정
MODELS_TO_COMPARE = [
    "gpt-4o-mini",
    "gpt-4.1-mini",
    "claude-3-5-sonnet-20240620",
    "claude-3-haiku-20240307"
]

def run_integrated_comparative_eval(results, heuristics_table):
    """
    GPT와 Claude 모델의 결과물을 가져와 전문가 관점에서 통합 비교 평가를 수행합니다.
    """
    # 평가자(Judge)는 가장 중립적이고 지능이 높은 GPT-4o를 사용합니다.
    evaluator_llm = ChatOpenAI(model="gpt-4o", temperature=0.1)
    
    # 평가 대상 데이터 구성
    candidates_text = ""
    for i, res in enumerate(results):
        # 섹션 2의 기술 통계 테이블에서 해당 모델 점수 매칭
        h_score = next((item for item in heuristics_table if item['모델'] == res['model']), {})
        
        candidates_text += f"""
---
[후보 {i+1}]
- 모델명: {res['model']}
- 기술 점수: 정확성 {h_score.get('정확성(0-1)', 'N/A')}, 형식 {h_score.get('형식(0-1)', 'N/A')}
- 답변 제목: {res['summary_title']}
- 답변 피드백: {res['scolding_message']}
- 답변 미션: {res['tomorrow_mission']}
"""

    # 고도화된 전략적 평가 프롬프트 (요청하신 대로 동일하게 유지)
    system_prompt = """
당신은 최고의 데이터 분석가이자 소비자 심리 전문가입니다. 
단순한 텍스트 비교를 넘어, 아래 4가지 고도화된 기준에 따라 모델들의 성능을 판별하세요.

1. 데이터 인사이트(Data Insight): 수치 간의 상관관계를 분석하여 소비 원인을 정확히 짚어내는가?
2. 전략적 넛지(Strategic Nudge): 사용자가 거부감 없이 즉각 실행할 수 있는 구체적이고 작은 행동 단위를 제시하는가?
3. 심리적 설득력(Persuasiveness): 사용자의 소비 페르소나를 반영한 맞춤형 톤앤매너를 구사하는가?
4. 구조적 명료성(Structural Clarity): 정보의 우선순위가 명확하여 분석 내용이 뇌에 즉각적으로 각인되는가?

결과는 반드시 아래 JSON 형식을 지켜주세요:
{
    "evaluation_criteria": ["데이터 인사이트", "전략적 넛지", "심리적 설득력", "구조적 명료성"],
    "ranking": [
        {"rank": 1, "model": "모델명", "reason": "선정 이유 (4가지 기준 근거)"},
        {"rank": 2, "model": "모델명", "reason": "..."},
        {"rank": 3, "model": "모델명", "reason": "..."},
        {"rank": 4, "model": "모델명", "reason": "..."}
    ],
    "final_verdict": "비즈니스 관점에서 최종 배포로 가장 적합한 모델 1개와 그 핵심 이유"
}
"""

    user_prompt = f"""
다음은 소비자 지출 분석 보고서에 대해 GPT와 Claude 모델들이 생성한 결과물입니다.
기술적 규칙 준수 여부와 분석의 질적 수준을 종합하여 엄격하게 순위를 매겨주세요.

{candidates_text}
"""

    print(f"⚖️  GPT vs Claude 통합 전략 분석 시작... (대상 모델: {len(results)}개)")
    
    try:
        response = evaluator_llm.invoke([
            ("system", system_prompt),
            ("user", user_prompt)
        ])
        
        # JSON 응답 파싱
        eval_result = json.loads(re.search(r"\{.*\}", response.content, re.DOTALL).group())
        
        # 시각적 결과 출력
        print("\n" + "="*80)
        print("📊 [LLM 통합 평가 보고서]")
        print("="*80)
        
        print("\n🔍 적용된 전문가 평가 기준:")
        for criterion in eval_result.get('evaluation_criteria', []):
            print(f" • {criterion}")
            
        print("\n🏆 모델별 최종 랭킹 및 분석:")
        for item in eval_result['ranking']:
            print(f" [{item['rank']}위] {item['model']}")
            print(f" └ 상세이유: {item['reason']}\n")
            
        print("="*80)
        print(f"💡 최종 권고: {eval_result['final_verdict']}")
        print("="*80)
        
        return eval_result

    except Exception as e:
        print(f"❌ 평가 과정 중 오류 발생: {e}")
        return None

